# KRATER Benchmark Data Analysis
---

post-benchmark exploratory analysis notebook.  
covers all metrics collected by KRATER: throughput, CPU, memory, cold start, throttling, and failure rates.

**dependencies:** `pip install pandas numpy matplotlib seaborn scipy`

### imports

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid', font_scale=1.1)

### config and constants

In [ ]:

RESULTS_SUBFOLDER = 'full_data'       # benchmark subfolder to analyze
RESULTS_BASE      = Path('../results')   

ENV_ORDER = ['debian', 'static', 'wasmtime', 'wasmer', 'wasmedge']

ENV_LABELS = {
    'krater-debian_default': 'debian',
    'krater-static_default': 'static',
    'krater-wasm_wasmtime':  'wasmtime',
    'krater-wasm_wasmer':    'wasmer',
    'krater-wasm_wasmedge':  'wasmedge',
}

PALETTE = {
    'debian':   '#4C72B0',
    'static':   '#DD8452',
    'wasmtime': '#55A868',
    'wasmer':   '#C44E52',
    'wasmedge': '#8172B2',
}

SIZES = [256, 1024]
PAIRS = [('wasmtime', 'debian'), ('wasmtime', 'static'), ('debian', 'static'), ('wasmer', 'debian')]

### data loading and processing

In [ ]:

def load_results(base: Path, subfolder: str) -> pd.DataFrame:
    folder = base / subfolder
    rows = []
    for json_path in sorted(folder.glob('*.json')): # iterate over all JSON files in the folder
        env_key = json_path.stem
        label   = ENV_LABELS.get(env_key, env_key)
        runtime = env_key.split('_', 1)[-1] # extract runtime from env_key (e.g. 'wasmtime' from 'krater-wasm_wasmtime')

        with open(json_path) as f:
            data = json.load(f)
            
        for run_key, trials in data.items(): # iterate over each run
            m = re.match(r'^(\d+)_(w|nw)$', run_key) # regex to extract size and warmup from run_key
            if not m:
                continue
            size   = int(m.group(1))
            warmup = m.group(2) == 'w'

            for t in trials: # iterate over each trial in the run
                pm = t.get('parsed_metrics', {})
                am = t.get('additional_metrics', {})
                ph = t.get('phases', {})
                rows.append({
                    'env':               env_key,
                    'label':             label,
                    'runtime':           runtime,
                    'size':              size,
                    'warmup':            warmup,
                    'trial':             t.get('trial'),
                    'valid':             pm.get('valid'),
                    'iterations':        pm.get('iterations'),
                    'throughput_mflops': pm.get('throughput_mflops'),
                    'cold_start_time':   am.get('cold_start_time'),
                    'avg_cpu_cores':     am.get('avg_cpu_cores'),
                    'peak_mem_bytes':    am.get('peak_mem_bytes'),
                    'avg_mem_bytes':     am.get('avg_mem_bytes'),
                    'throttled_events':  am.get('throttled_events'),
                    'phase_start':       ph.get('start'),
                    'phase_running':     ph.get('running_time'),
                    'phase_bench_start': ph.get('bench_start'),
                    'phase_end':         ph.get('end'),
                    'samples':           t.get('samples', []),
                })

    df = pd.DataFrame(rows)
    df['label'] = pd.Categorical(df['label'], categories=ENV_ORDER, ordered=True) # make label categorical
    return df.sort_values(['label', 'size', 'trial']).reset_index(drop=True) # sort by label, size, and trial


df = load_results(RESULTS_BASE, RESULTS_SUBFOLDER)
print(f"Loaded {len(df)} trial entries from '{RESULTS_SUBFOLDER}'")
df.head(5)

### data quality check

In [ ]:
quality = (
    df.groupby(['label', 'size'], observed=True)
    .agg(
        total          = ('valid', 'count'),
        valid_trials   = ('valid', lambda x: x.eq(True).sum()),
        invalid_trials = ('valid', lambda x: (~x.eq(True)).sum()),
        no_mflops      = ('throughput_mflops', lambda x: x.isna().sum()),
        no_cold_start  = ('cold_start_time',   lambda x: x.isna().sum()),
    )
    .reset_index()
)
display(quality)


---
## 1. compute throughput (MFLOPS)

### calculate mean, std. deviation, ci95, and coefficient of variation

In [ ]:
valid_df = df[df['valid'] == True].copy() # filter invalid trials

def ci95(x): # helper to calculate 95% CI for a series of values
    n = len(x)
    return stats.t.ppf(0.975, n - 1) * x.sem() if n > 1 else np.nan # t-distribution multiplier times standard error of the mean, if n > 1, apparently

mflops_stats = ( # calculate stats for throughput_mflops by label and size
    valid_df.groupby(['label', 'size'], observed=True)['throughput_mflops']
    .agg(
        n      = 'count',
        mean   = 'mean',
        std    = 'std',
        cv_pct = lambda x: x.std() / x.mean() * 100, # coeff of variation, as a percentage of the mean
        ci_95  = ci95,
    )
    .round(3)
    .reset_index()
)
display(mflops_stats)

In [ ]:
# two-panel figure per matrix size:
#   top row:    all 5 environments on log scale (for wasmedge gap)
#   bottom row: JIT/native only on linear scale (for fine differences)

competitive = [e for e in ENV_ORDER if e != 'wasmedge']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for col, size in enumerate(SIZES):
    sub = valid_df[valid_df['size'] == size]

    ax = axes[0][col]
    sns.boxplot(data=sub, x='label', y='throughput_mflops',
                order=ENV_ORDER, palette=PALETTE, hue='label', ax=ax, width=0.5) # plot all environments on log scale
    ax.set_yscale('log')
    ax.set_title(f'N={size} — all environments (log scale)')
    ax.set_xlabel('')
    ax.set_ylabel('MFLOPS (log)')
    ax.set_ylim(bottom=10)
    ax.tick_params(axis='x')

    ax = axes[1][col]
    sub_c = sub[sub['label'].isin(competitive)]
    sns.boxplot(data=sub_c, x='label', y='throughput_mflops',
                order=competitive, palette=PALETTE, hue='label', ax=ax, width=0.5) # plot JIT/native only on linear scale
    ax.set_title(f'N={size} — JIT/native only (linear scale)')
    ax.set_xlabel(''); ax.set_ylabel('MFLOPS')
    ax.tick_params(axis='x')

plt.suptitle('Compute Throughput (MFLOPS)', fontsize=14)
plt.tight_layout()
plt.show()

### normality check, test selection, pairwise comparisons, holm-bonferroni correction

In [ ]:
# normality check: shapiro-wilk test for each group (label, size). requires at least 3 samples to run.
normality_results = []
for (label, size), grp in valid_df.groupby(['label', 'size'], observed=True):
    x = grp['throughput_mflops'].dropna().values # get the throughput values for this group, dropping NaNs
    if len(x) >= 3:
        stat, p = stats.shapiro(x)
        normality_results.append({
            'label': label, 'size': size,
            'W': round(float(stat), 4), # W statistic from the Shapiro-Wilk test, where values close to 1 suggest normality
            'p': round(float(p), 4), # p-value from the test, where values above 0.05 suggest normality
            'normal': bool(p > 0.05),
        })

norm_df = pd.DataFrame(normality_results)
display(norm_df)

# choose test based on normality: if all groups are normal use Welch's t-test. otherwise use Mann-Whitney U test
all_normal = norm_df['normal'].all()
test_name  = "Welch's t-test" if all_normal else "Mann-Whitney U"
print(f"\nAll groups normal: {all_normal}  → using {test_name}")


# cohen's d measures how many standard deviations the means of two groups are apart. |d| >0.8 is considered a large effect size
# this complements the tests by quantifying the magnitude of the difference, not just its significance
def cohens_d(a: pd.Series, b: pd.Series) -> float:
    n1, n2   = len(a), len(b)
    pooled   = np.sqrt(((n1 - 1) * a.std()**2 + (n2 - 1) * b.std()**2) / (n1 + n2 - 2))
    return float((a.mean() - b.mean()) / pooled) if pooled > 0 else np.nan


# pairwise comparisons
raw_results = []
for size in SIZES:
    sub = valid_df[valid_df['size'] == size]
    for a, b in PAIRS:
        ga = sub[sub['label'] == a]['throughput_mflops'].dropna()
        gb = sub[sub['label'] == b]['throughput_mflops'].dropna()
        if len(ga) < 2 or len(gb) < 2:
            continue
        if all_normal:
            stat, p = stats.ttest_ind(ga, gb, equal_var=False)
        else:
            stat, p = stats.mannwhitneyu(ga, gb, alternative='two-sided')
        raw_results.append({
            'env_a':     a, 'env_b': b, 'size': size,
            'test_used': test_name,
            'statistic': round(float(stat), 4),
            'p_raw':     round(float(p), 7),
            'cohens_d': round(cohens_d(ga, gb), 4),
        })

# holm-bonferroni correction
raw_results.sort(key=lambda x: x['p_raw'])
m = len(raw_results)
running_max = 0.0
for k, r in enumerate(raw_results):
    p_corr = min(r['p_raw'] * (m - k), 1.0) # apply correction
    p_corr = max(p_corr, running_max) # enforce monotonicity
    running_max = p_corr
    r['p_corrected'] = round(p_corr, 6)
    r['significant'] = p_corr < 0.05 # if samples are few, it may be impossible to achieve significance after correction. i recommend running with 25 samples

test_results = pd.DataFrame(raw_results).sort_values(['size', 'env_a', 'env_b']).reset_index(drop=True)
column_order = [
    'size', 'env_a', 'env_b', 'test_used', 
    'statistic', 'p_raw', 'p_corrected', 
    'significant', 'cohens_d'
]
test_results = test_results[column_order]
display(test_results)

---
## 2. MFLOPS scaling: N=256 -> N=1024

### pivot table

In [ ]:
pivot = ( # create a pivot table of mean MFLOPS by label and size
    valid_df.groupby(['label', 'size'], observed=True)['throughput_mflops']
    .mean()
    .unstack('size')
)
pivot.columns = [f'mflops_{c}' for c in pivot.columns]
pivot['ratio_1024_to_256'] = (pivot['mflops_1024'] / pivot['mflops_256']).round(4)
display(pivot.round(4))

### mflops scaling bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE.get(str(l), '#888888') for l in pivot.index] 
pivot['ratio_1024_to_256'].plot(kind='bar', ax=ax, color=colors, width=0.6, edgecolor='white')
ax.bar_label(ax.containers[0], fmt='%.3f', label_type='center', color='white')
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--', label='no change')
ax.set_title('MFLOPS Scaling: N=1024 / N=256')
ax.set_xlabel(''); ax.set_ylabel('Ratio')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

---
## 3. CPU utilization

### average cpu utilisation boxplots

In [ ]:
cpu_df = valid_df.dropna(subset=['avg_cpu_cores'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=cpu_df, x='label', y='avg_cpu_cores', hue='size',
            palette='Set2', ax=ax, order=ENV_ORDER)
ax.set_title('Average CPU Utilization (1.0 = 1 full core saturated)')
ax.set_xlabel(''); ax.set_ylabel('avg_cpu_cores')
plt.tight_layout()
ax.set_ylim(bottom=0.998, top=1)
plt.show()

### kernel time fraction (system / total)

In [ ]:
# system_usec delta / total_usec delta across the trial window
# usage_usec and system_usec are cumulative, diff first and last sample

def kernel_fraction(samples):
    if len(samples) < 2:
        return np.nan
    total  = samples[-1]['usage_usec']  - samples[0]['usage_usec'] # delta of usage_usec from first to last sample
    system = samples[-1]['system_usec'] - samples[0]['system_usec'] # delta of system_usec from first to last sample
    return system / total if total > 0 else np.nan

valid_df['kernel_fraction'] = valid_df['samples'].apply(kernel_fraction)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=valid_df, x='label', y='kernel_fraction', hue='size', palette='Set2', ax=ax, order=ENV_ORDER, errorbar='sd')
ax.set_title('Kernel Time Fraction (system_usec / total_usec)')
ax.set_xlabel('')
ax.set_ylabel('Fraction')
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))
plt.show()

### cpu utilisation time series

In [ ]:
def plot_cpu_utilization(df, trial_id, n_size, env_order, palette, x_limit=40):
    subset = df[(df['size'] == n_size) & (df['trial'] == trial_id)]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    plotted_any = False

    for label in env_order:
        row = subset[subset['label'] == label] # first filter for the specific label in the subset
        samples = row.iloc[0]['samples'] # then get the samples for this label, trial, and size

        t0 = samples[0]['timestamp'] # normalize timestamps to start at 0
        ts = [s['timestamp'] - t0 for s in samples[1:]]
        
        # CPU calculation: delta usage / delta time * 1e6
        cpu = [
            (samples[i + 1]['usage_usec'] - samples[i]['usage_usec']) /
            ((samples[i + 1]['timestamp'] - samples[i]['timestamp']) * 1_000_000)
            for i in range(len(samples) - 1)
        ]

        ax.plot(ts, cpu, label=label, color=palette.get(label, None))
        plotted_any = True
            

    if not plotted_any:
        print("No valid data points were found to plot.")
        plt.close(fig)
        return

    ax.set_title(f'Instantaneous CPU Utilization — Trial {trial_id}, N={n_size}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('CPU cores')
    ax.set_xlim(0, x_limit)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_cpu_utilization(valid_df, trial_id=13, n_size=256, env_order=ENV_ORDER, palette=PALETTE)
plot_cpu_utilization(valid_df, trial_id=4, n_size=1024, env_order=ENV_ORDER, palette=PALETTE)

---
## 4. Memory Footprint

### peak mem bytes / avg mem bytes bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (col, title) in zip(axes, [
    ('peak_mem_bytes', 'Peak Memory (MB)'),
    ('avg_mem_bytes',  'Average Memory (MB)'),
]):
    tmp = valid_df.dropna(subset=[col]).copy()
    tmp[col] = tmp[col] / 1024**2
    sns.barplot(data=tmp, x='label', y=col, hue='size',
                palette='Set2', ax=ax, order=ENV_ORDER, errorbar='sd')
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('MB')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Memory Footprint', fontsize=13)
plt.tight_layout()
plt.show()

### memory time series

In [ ]:
def plot_memory(df, trial_id, n_size, mode='usage'):
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # theoretical baseline in bytes
    # 3 matrices * N^2 * 8 bytes (double precision)
    baseline_bytes = 3 * (n_size**2) * 8
    
    for label in ENV_ORDER:
        row = df[(df['label'] == label) & 
                 (df['size']  == n_size) & 
                 (df['trial'] == trial_id)]
        
        if row.empty:
            continue
            
        samples = row.iloc[0]['samples']
        if not samples:
            continue
            
        t0 = samples[0]['timestamp']
        ts = [s['timestamp'] - t0 for s in samples]
        
        if mode == 'overhead':
            values = [(s['mem_bytes'] - baseline_bytes) / 1024**2 for s in samples]
            ylabel = 'Memory Overhead (MB)'
            title = f'Memory Overhead — Trial {trial_id}, N={n_size}'
        else:
            values = [s['mem_bytes'] / 1024**2 for s in samples]
            ylabel = 'mem_bytes (MB)'
            title = f'Memory Usage Over Time — Trial {trial_id}, N={n_size}'
            
        ax.plot(ts, values, label=label, color=PALETTE[label])

    ax.set_title(title)
    ax.set_xlabel('Time since bench_start (s)')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.set_xlim(0, 40)

    plt.show()

plot_memory(valid_df, trial_id=5, n_size=256, mode='usage')
plot_memory(valid_df, trial_id=5, n_size=256, mode='overhead')
plot_memory(valid_df, trial_id=13, n_size=1024, mode='usage')
plot_memory(valid_df, trial_id=13, n_size=1024, mode='overhead')

---
## 5. Pod Cold Start Latency

In [ ]:
cold_df = valid_df.dropna(subset=['cold_start_time'])

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=cold_df, x='label', hue='label', y='cold_start_time',
            order=ENV_ORDER, palette=PALETTE, ax=ax, width=0.5, dodge=False, legend=False)
ax.set_title('Pod Cold Start Latency  (running_time − start)')
ax.set_xlabel(''); ax.set_ylabel('Seconds')
plt.tight_layout()
plt.show()

cold_stats = (
    cold_df.groupby('label', observed=True)['cold_start_time']
    .agg(n='count', mean='mean', std='std', min='min', max='max')
    .round(3)
    .reset_index()
)
display(cold_stats)

---
## 6. CPU Throttling

In [ ]:
thr_df = valid_df.dropna(subset=['throttled_events'])

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=thr_df, x='label', y='throttled_events', hue='size',
            palette='Set2', ax=ax, order=ENV_ORDER)
ax.set_title('CPU Throttle Events per Trial')
ax.set_xlabel(''); ax.set_ylabel('throttled_events')
plt.tight_layout()
plt.show()

---
## 7. Runtime Failure Rates (Clopper-Pearson)

In [ ]:
cp_rows = []

for (label, size), grp in df.groupby(['label', 'size'], observed=True):
    n = len(grp)
    k = int((~grp['valid'].eq(True)).sum())

    lower = float(stats.beta.ppf(0.025, k,     n - k + 1)) if k > 0 else 0.0
    upper = float(stats.beta.ppf(0.975, k + 1, n - k))     if k < n else 1.0

    cp_rows.append({
        'label':        label,
        'size':         size,
        'n':            n,
        'k':            k,
        'failure_rate': k / n,
        'ci_lower':     lower,
        'ci_upper':     upper,
    })

cp_df = pd.DataFrame(cp_rows)
display(cp_df.round(4))

fig, axes = plt.subplots(1, len(SIZES), figsize=(12, 5), sharey=True)
for ax, size in zip(axes, SIZES):
    sub     = cp_df[cp_df['size'] == size].reset_index(drop=True)
    labels  = [str(l) for l in sub['label']]
    rates   = sub['failure_rate'].values
    errs_lo = rates - sub['ci_lower'].values
    errs_hi = sub['ci_upper'].values - rates
    colors  = [PALETTE.get(l, '#888') for l in labels]
    ax.barh(labels, rates, xerr=[errs_lo, errs_hi],
            color=colors, capsize=4, alpha=0.85)
    ax.set_title(f'N={size}')
    ax.set_xlabel('Failure rate')
    ax.axvline(0, color='black', linewidth=0.5)

plt.suptitle('Failure Rates with 95% Clopper-Pearson CI', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Summary Table

In [ ]:
summary = (
    valid_df.groupby(['size', 'label'], observed=True)
    .agg(
        n              = ('throughput_mflops', 'count'),
        mflops_mean    = ('throughput_mflops', 'mean'),
        mflops_std     = ('throughput_mflops', 'std'),
        cold_start_s   = ('cold_start_time',   'mean'),
        avg_cpu        = ('avg_cpu_cores',      'mean'),
        peak_mem_mb    = ('peak_mem_bytes',     lambda x: x.mean() / 1024**2),
        avg_mem_mb     = ('avg_mem_bytes',      lambda x: x.mean() / 1024**2),
        throttled_mean = ('throttled_events',   'mean'),
    )
    .round(2)
    .reset_index()
)
display(summary)